## Imports

In deze eerste stap importeren we alle libraries die nodig zijn voor het project. TensorFlow/Keras wordt gebruikt voor het deep learning model, NumPy en Pandas voor data, Matplotlib voor grafieken en Scikit-learn voor controles zoals de confusion matrix.

In [ ]:
# TensorFlow and tf.keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import optimizers
from tensorflow.keras import layers
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
# DenseNet alternative, if you want to test it later:
# from tensorflow.keras.applications import DenseNet121
# from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.callbacks import Callback, ReduceLROnPlateau

# helper libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
import pandas as pd
from PIL import Image
import os

## Load data

Hier laden we de training labels uit `train.csv` en maken we een train/validation split. Daarna worden de beelden ingeladen via `image_dataset_from_directory`, omdat de `train` folder al per klasse is opgedeeld. Zo kan Keras de labels automatisch afleiden uit de mapnamen.

In [ ]:
# version with numpy array's for easier viewing of data
SEED = 42
IMG_SIZE = 384
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

train_data = np.genfromtxt('train.csv', delimiter=',', skip_header=1, dtype='str')

train_x = train_data[:, 0]
train_y = train_data[:, 1].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    train_x,
    train_y,
    test_size=0.2,
    # random_state=SEED,
    stratify=train_y
)

raw_train_dataset = image_dataset_from_directory(
    "train",
    labels="inferred",
    label_mode="int",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset="training",
    seed=SEED
)

raw_validation_dataset = image_dataset_from_directory(
    "train",
    labels="inferred",
    label_mode="int",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset="validation",
    seed=SEED
)

class_names = raw_train_dataset.class_names
NUM_CLASSES = len(class_names)
print(class_names)

train_dataset = raw_train_dataset.cache().shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
validation_dataset = raw_validation_dataset.cache().prefetch(AUTOTUNE)

train_datatset = train_dataset
test_dataset = validation_dataset

# Class weights: hogere waarde = model besteedt meer aandacht aan die klasse
class_weight_dict = {
    0: 1.0,   # Black_spiny_tailed_iguana  → goed
    1: 1.5,   # Brown_anole                → matig
    2: 1.0,   # Cuban_knight_anole         → goed
    3: 1.5,   # Desert_iguana              → goed
    4: 1.5,   # Green_anole                → slecht
    5: 1.5,   # Green_iguana               → slecht
    6: 1.0,   # Lesser_Antillean_iguana    → goed
}

## Check labels

Voor we trainen controleren we welke labels er zijn en of de verdeling tussen training en validation ongeveer logisch blijft. Dit helpt om te zien of de data niet scheef verdeeld is.

<table>
<tr><th>Value</th><th>Class</th></tr>
<tr><td>0</td>	<td>Black_spiny_tailed_iguana</td></tr>
<tr><td>1</td>	<td>Brown_anole</td></tr>
<tr><td>2</td>	<td>Cuban_knight_anole</td></tr>
<tr><td>3</td>	<td>Desert_iguana</td></tr>
<tr><td>4</td>	<td>Green_anole</td></tr>
<tr><td>5</td>	<td>Green_iguana</td></tr>
<tr><td>6</td>	<td>Lesser_Antillean_iguana</td></tr>
</table>


In [ ]:
y_test_unique, y_test_count = np.unique(y_test,return_counts=True)
y_train_unique, y_train_count = np.unique(y_train,return_counts=True)
print(y_test_unique)
print(y_test_count)
print(y_train_unique)
print(y_train_count)

We printen de eerste en laatste labels van de train en validation set. Zo kunnen we snel controleren of de split er normaal uitziet en niet per ongeluk alle gelijke labels bij elkaar staan.

In [ ]:
# Print the first and last 10 labels from train/test set
print(y_test[:10])
print(y_test[-10:])
print(y_train[:10])
print(y_train[-10:])

Hier kan je een paar voorbeeldbeelden tonen. Dit is vooral handig om visueel te controleren of de afbeeldingen correct worden ingeladen en of de klassen ongeveer kloppen.

In [ ]:
# plt.figure(figsize=(10,10))
# for i in range(10):
#     plt.subplot(4,5,i+1)
#     plt.xticks([])
#     plt.yticks([])
#     plt.grid(False)
#     plt.imshow(X_train[i],cmap='binary')
#     plt.xlabel(class_names[y_train[i]])

## Define model settings

In dit blok bouwen we het model. In plaats van een volledig CNN vanaf nul te trainen, gebruiken we transfer learning: een pretrained model haalt beeldkenmerken uit de foto, en onze eigen laatste lagen leren de 7 iguana/anole klassen herkennen.

In [ ]:
# # Define your sequential model, with the different layers, including the preprocessing layers
# model =keras.Sequential([
#     # Preprocessing: Add a Rescaling layer to rescale the pixel values to the [0, 1] range
#     layers.Rescaling(1./255),
#     # Input Layer: Add a Flattening layer to make 1-D vector of our 28x28 images
#     layers.Flatten(input_shape=(128, 128)),
#     # Hidden layer: Add a Dense layer, aka a fully or densely connected layer of 128 neurons, and let them use the 'ReLu'-squishing or activation function
#     layers.Dense(128, activation='relu'),
    
#     # Extra hidden layers
#     layers.Dense(512, activation='relu'),


#     # dropout layer applies to the layer above it 
#     # dropout disabels the selected amount(25%) of neurons during one run of the model to decrease the reliance on specific neurons
#     keras.layers.Dropout(0.25), 
#     layers.Dense(512, activation='relu'),
#     # dropout layer applies to the layer above it
#     keras.layers.Dropout(0.25),  
#     layers.Dense(256, activation='sigmoid'),
#     layers.Dense(128, activation='sigmoid'),
    
#     # Output layer: Add a Dense layer of 10 neurons (because we have 10 possible output labels), and link those neurons together in a group, via the 'softmax'-activation function
#     layers.Dense(7, activation='softmax')
# ]) 

Wij hebben geprobeerd met `EfficientNetV2S` en `DenseNet121`. DenseNet was voor kleiner dataset dus dit was de eerste optie, maar omdat de accuracy erdoor lager was hebben we gekozen voor efficientNet

In [ ]:
# Transfer learning with EfficientNetV2S.
# This follows the transfer-learning example: frozen pretrained body + GlobalAveragePooling2D + simple Dense head.
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.04),
    layers.RandomZoom(0.08),
    layers.RandomContrast(0.2),
], name="data_augmentation")

base_model = EfficientNetV2S(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)


# Alternative with DenseNet121:
# base_model = DenseNet121(
#     include_top=False,
#     weights="imagenet",
#     input_shape=(IMG_SIZE, IMG_SIZE, 3)
# )
# If you use DenseNet121, use the DenseNet preprocess_input import instead.

base_model.trainable = False

model = keras.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    data_augmentation,
    layers.Lambda(preprocess_input, name="efficientnetv2_preprocess"),
    base_model,
    layers.GlobalAveragePooling2D(),
    
    layers.Dense(128, activation='relu'),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

## Compile model

Hier kiezen we hoe het model gaat leren. We gebruiken `sparse_categorical_crossentropy` omdat de labels gewone getallen zijn, en `accuracy` om te volgen hoeveel beelden correct geclassificeerd worden.

In [ ]:
# Compile the model with the pretrained EfficientNetV2S body frozen.
# We only train the new classification head, which is more stable on this small dataset.
from tensorflow.keras.metrics import F1Score

class SparseF1(tf.keras.metrics.Metric):
    def __init__(self, name='f1', **kwargs):
        super().__init__(name=name, **kwargs)
        self.f1 = F1Score(average='macro')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.one_hot(tf.cast(y_true, tf.int32), depth=tf.shape(y_pred)[-1])
        self.f1.update_state(y_true, y_pred, sample_weight)

    def result(self):
        return self.f1.result()

    def reset_states(self):
        self.f1.reset_states()

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=optimizers.Adam(learning_rate=0.001),
    metrics=[SparseF1(), "accuracy"]
)

## Train model

In deze stap trainen we het model op de training set en testen we na elke epoch op de validation set. Zo kunnen we volgen of het model echt beter wordt, of alleen de trainingbeelden vanbuiten begint te leren.

Wij gebruiken hier `RestoreBestValidationWeights(Callback):`. dit zorgt ervoor dat hij de beste validation accuracy per epoch pakt. dus al zou hij die 100 keer testen pakt die gwn de beste die erin staat. dit zal ervoor zorgen dat ons model de weights gebruikt van de epoch die de beste `val_accuracy`


In [ ]:
# Train for exactly 8 epochs.
# Validation accuracy can dip from one epoch to the next, so this callback keeps the best weights in memory.
class RestoreBestValidationWeights(Callback):
    def __init__(self, monitor='val_accuracy'):
        super().__init__()
        self.monitor = monitor
        self.best_value = -np.inf
        self.best_weights = None
        self.best_epoch = 0

    def on_epoch_end(self, epoch, logs=None):
        current_value = logs.get(self.monitor)
        if current_value is not None and current_value > self.best_value:
            self.best_value = current_value
            self.best_weights = self.model.get_weights()
            self.best_epoch = epoch + 1

    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)
            print(f"Restored epoch {self.best_epoch} weights with best {self.monitor}: {self.best_value:.4f}")

callbacks = [
    # Verlaagt de learning rate automatisch als val_f1 niet meer verbetert.
    # Dit helpt het model om fijner te leren wanneer het vast lijkt te zitten.
    ReduceLROnPlateau(
        monitor='val_f1',      # kijkt naar de validation F1-score
        factor=0.3,            # elke keer dat het triggert: learning rate × 0.3 (dus 70% kleiner)
        patience=2,            # wacht 2 epochs zonder verbetering voordat het ingrijpt
        min_lr=1e-6,            # de learning rate mag niet kleiner worden dan 0.000001
        mode="max"
    ),
    RestoreBestValidationWeights(monitor='val_f1')
]

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=20,
    callbacks=callbacks
)

## Validation

Na het trainen bekijken we de loss en accuracy curves. De training curve toont hoe goed het model leert op de trainingsdata. De validation curve is belangrijker, omdat die beter toont hoe goed het model generaliseert naar nieuwe beelden.

In [ ]:
def plotLosses(history):
  # Create a figure and a grid of subplots with a single call
  fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10,5))
  # Plot the loss curves on the first subplot
  ax1.plot(history.history['loss'], label='training loss')
  ax1.plot(history.history['val_loss'], label='validation loss')
  ax1.set_title('Loss curves')
  ax1.set_xlabel('Epoch')
  ax1.set_ylabel('Loss')
  ax1.legend()
  # Plot the accuracy curves on the second subplot
  ax2.plot(history.history['accuracy'], label='training accuracy')
  ax2.plot(history.history['val_accuracy'], label='validation accuracy')
  ax2.set_title('Accuracy curves')
  ax2.set_xlabel('Epoch')
  
  ax2.set_ylabel('Accuracy')
  ax2.legend()
  # Adjust the spacing between subplots
  fig.tight_layout()
  # Show the figure
  plt.show()


plotLosses(history)

In [ ]:
model.summary()

## Prediction

In dit laatste deel evalueren we het model, bekijken we waar het fout gaat met een confusion matrix, en maken we daarna een submission bestand voor de testbeelden.

In [ ]:
test_loss, test_acc, test_f1 = model.evaluate(test_dataset)

print('Validation accuracy:', test_acc)

### Confusion matrix

De confusion matrix toont per echte klasse welke voorspelling het model maakt. We gebruiken hier de labelnummers `0` tot `6`, zodat de grafiek leesbaar blijft. De tabel hierboven toont welk nummer bij welke klasse hoort.

In [ ]:
# Confusion matrix: shows which classes the model confuses with each other.
y_true = []
y_pred = []

for images, labels in validation_dataset:
    predictions = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

cm = confusion_matrix(y_true, y_pred)

# Use numbers 0-6 in the plot, so the matrix stays readable.
# The table above shows which number belongs to which class.
label_numbers = np.arange(NUM_CLASSES)

plt.figure(figsize=(8, 8))
display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_numbers)
display.plot(cmap="Blues", values_format="d")
plt.title("Confusion matrix")
plt.show()

# Extra overview: print the most common mistakes.
mistakes = []
for true_class in range(NUM_CLASSES):
    for predicted_class in range(NUM_CLASSES):
        if true_class != predicted_class and cm[true_class, predicted_class] > 0:
            mistakes.append((cm[true_class, predicted_class], true_class, predicted_class))

mistakes = sorted(mistakes, reverse=True)
for count, true_label, predicted_label in mistakes[:10]:
    print(f"{count}x: label {true_label} predicted as label {predicted_label}")

### Test predictions

Hier voorspellen we de echte testbeelden. De volgorde wordt uit `test.csv` gehaald, zodat de voorspellingen in dezelfde volgorde staan als verwacht wordt voor de submission.

In [ ]:
# Generate predictions for the real test images in the same order as test.csv.
test_ids = pd.read_csv("test.csv")["id"].astype(str).tolist()
test_paths = [os.path.join("test", f"{image_id}.jpg") for image_id in test_ids]

def load_test_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    return tf.cast(image, tf.float32)

test_prediction_dataset = (
    tf.data.Dataset.from_tensor_slices(test_paths)
    .map(load_test_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

predictions = model.predict(test_prediction_dataset)
pred = np.argmax(predictions, axis=1)

### Submission file

In deze laatste cel maken we het CSV-bestand met de kolommen `id` en `label`. Dit bestand kan gebruikt worden als submission.

In [ ]:
# Match the format of sample_submission.csv: id,label
df = pd.DataFrame(data={"id": test_ids, "label": pred.flatten()})
df.to_csv("test_submission.csv", index=False)
df.head()